# Week 2 Day 03 - Working with some other models and guardrails

#### Part 01: Different Models
#### Part 02: Structured Outputs
#### Part 03: Guardrails

### Part 01: Different Models (but we probably wil use Azure OpenAI models only)

In [3]:
import os
import requests
from dotenv import load_dotenv
from openai.types.responses import ResponseTextDeltaEvent
from agents import Agent, Runner, trace, function_tool, SQLiteSession, OpenAIChatCompletionsModel, set_tracing_disabled, output_guardrail, GuardrailFunctionOutput
from openai import AsyncAzureOpenAI 
from anthropic import AsyncAnthropicFoundry
from agents.model_settings import ModelSettings
from IPython.display import Markdown, display
from agents.extensions.visualization import draw_graph
import asyncio
from typing import Dict, Any, List, Optional, Union
import datetime
import win32com.client
from pydantic import BaseModel, Field
import pythoncom
import subprocess
import re
import time

set_tracing_disabled(True)  # Disable tracing for this notebook to avoid cluttering the output
load_dotenv(override=True) # Load environment variables from .env file, override existing ones if necessary

True

In [4]:
# Let's see if the API key is working/helping us to call LLM from Azure Foundry
import os
# From OpenAI
AZURE_OPENAI_API_KEY= os.getenv("AZURE_OPENAI_API_KEY")
AZURE_OPENAI_MODEL_DEPLOYMENT = os.getenv("AZURE_OPENAI_DEPLOYMENT")
AZURE_OPENAI_ENDPOINT = os.getenv("AZURE_OPENAI_ENDPOINT")
AZURE_OPENAI_API_VERSION = os.getenv("AZURE_OPENAI_API_VERSION")
AZURE_OPENAI_DEPLOYMENT_GPT_41 = os.getenv("AZURE_OPENAI_DEPLOYMENT_GPT_41")
AZURE_OPENAI_DEPLOYMENT_GPT_54_mini = os.getenv("AZURE_OPENAI_DEPLOYMENT_GPT_54_mini")
AZURE_OPENAI_DEPLOYMENT_GPT_55 = os.getenv("AZURE_OPENAI_DEPLOYMENT_GPT_55")
AZURE_OPENAI_DEPLOYMENT_GPT_4O_mini = os.getenv("AZURE_OPENAI_DEPLOYMENT_GPT_4O_mini")
if AZURE_OPENAI_API_KEY:
    print("AZURE_OPENAI_API_KEY is available")
else:
    print("AZURE_OPENAI_API_KEY is not available")

# From Anthropic
AZURE_CLAUDE_DEPLOYMENT_OPUS_48=os.getenv("AZURE_CLAUDE_DEPLOYMENT_OPUS_48")
AZURE_CLAUDE_ENDPOINT=os.getenv("AZURE_CLAUDE_ENDPOINT")
AZURE_CLAUDE_API_KEY=os.getenv("AZURE_CLAUDE_API_KEY")
if AZURE_CLAUDE_API_KEY:
    print("AZURE_CLAUDE_API_KEY is avaiable")
else:
    print("AZURE_CLAUDE_API_KEY is not available")

# Loading Email addresses
EMAIL_ADDRESS = os.getenv("EMAIL_ADDRESS_TO")
if EMAIL_ADDRESS: 
    print("Email address found!")
else:
    print("Email address not found!")

AZURE_OPENAI_API_KEY is available
AZURE_CLAUDE_API_KEY is avaiable
Email address found!


In [5]:
## setting up the client for OpenAI using Azure Endpoint (Azure Foundry)
client_openai = AsyncAzureOpenAI(
    azure_endpoint=AZURE_OPENAI_ENDPOINT,
    api_key=AZURE_OPENAI_API_KEY,
    api_version=AZURE_OPENAI_API_VERSION
)
client_anthropic = AsyncAnthropicFoundry(
    base_url=AZURE_CLAUDE_ENDPOINT,
    api_key=AZURE_CLAUDE_API_KEY
)


## let's a define a model pointing to Azure deployment
model_openai_GPT_54 = OpenAIChatCompletionsModel(
    openai_client=client_openai,
    model=AZURE_OPENAI_DEPLOYMENT_GPT_54_mini
)
model_openai_GPT_41 = OpenAIChatCompletionsModel(
    openai_client=client_openai,
    model=AZURE_OPENAI_DEPLOYMENT_GPT_41
)
model_openai_CLAUDE_OPUS_48 = OpenAIChatCompletionsModel(
    openai_client=client_anthropic,
    model=AZURE_CLAUDE_DEPLOYMENT_OPUS_48
)

In [6]:
## Setting up the intro-prompt
# Let's set up a system prompt
intro = """
You are a sales agent working for a company called 'CommodiPulse',
a company which delivers lag-free, real-time data feeds for global commodities including power, carbon, agriculture, oil, and gas.
We empower businesses with instant, synchronized market insights to make critical trading and supply decisions with absolute certainty.
"""

instructions = intro + "You write compelling sales emails that are likely to get response."


In [7]:
# let's define agents as tools with above defined clients
tool_description = "Use this tool to write a sales email. In input, just instruct it to write a sales email."
AI_MODEL_CLIENTS = {
    "GPT_41_Agent":model_openai_GPT_41,
    "GPT_54_Agent":model_openai_GPT_54, 
    "CLUADE_OPUS_48":model_openai_CLAUDE_OPUS_48
}
tools = []
for model_name, client in AI_MODEL_CLIENTS.items():
    tool = Agent(
        name = model_name, 
        instructions = instructions,
        model = client
    ).as_tool(
        tool_name=model_name,
        tool_description=tool_description
    )
    # adding each created tool in a tools list
    tools.append(tool)

print(tools)


[FunctionTool(name='GPT_41_Agent', description='Use this tool to write a sales email. In input, just instruct it to write a sales email.', params_json_schema={'description': 'Default input schema for agent-as-tool calls.', 'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'AgentAsToolInput', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<agents.tool._FailureHandlingFunctionToolInvoker object at 0x000002AFF97E5780>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None, needs_approval=False, timeout_seconds=None, timeout_behavior='error_as_result', timeout_error_function=None, defer_loading=False, custom_data_extractor=None), FunctionTool(name='GPT_54_Agent', description='Use this tool to write a sales email. In input, just instruct it to write a sales email.', params_json_schema={'description': 'Default input schema for agent-as-tool calls.', 'properties': {'input': {'title': 'In

In [8]:
import importlib
import sys

# Force reload of messenger to pick up any file changes cached in the kernel
if "messenger" in sys.modules:
    importlib.reload(sys.modules["messenger"])

from messenger import send_email_tool, record_message_tool

# send_email_tool(
#     "Lab03_Test",
#     "Hello there!"
# )


In [9]:
### let's define a function tool with above imported function
@function_tool
def send_email__tool(
    subject: str,
    text_body: str,
    ) -> str:
    """
    Send out an email with the given subject and body to all sales prospects

    Args:
        subject: The subject of the email
        text_body: the body of the email as plain text
    """
    email_sent = send_email_tool(
        subject = subject,
        body = text_body
    )
    if email_sent:
        return "Email sent successfully"
    else:
        return "********### FAILED TO SEND EMAIL ###**********"
    

In [27]:
tools

[FunctionTool(name='GPT_41_Agent', description='Use this tool to write a sales email. In input, just instruct it to write a sales email.', params_json_schema={'description': 'Default input schema for agent-as-tool calls.', 'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'AgentAsToolInput', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<agents.tool._FailureHandlingFunctionToolInvoker object at 0x0000020A6D105810>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None, needs_approval=False, timeout_seconds=None, timeout_behavior='error_as_result', timeout_error_function=None, defer_loading=False, custom_data_extractor=None),
 FunctionTool(name='GPT_54_Agent', description='Use this tool to write a sales email. In input, just instruct it to write a sales email.', params_json_schema={'description': 'Default input schema for agent-as-tool calls.', 'properties': {'input': {'title': 'I

In [10]:
tools.append(send_email__tool)
tools

[FunctionTool(name='GPT_41_Agent', description='Use this tool to write a sales email. In input, just instruct it to write a sales email.', params_json_schema={'description': 'Default input schema for agent-as-tool calls.', 'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'AgentAsToolInput', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<agents.tool._FailureHandlingFunctionToolInvoker object at 0x000002AFF97E5780>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None, needs_approval=False, timeout_seconds=None, timeout_behavior='error_as_result', timeout_error_function=None, defer_loading=False, custom_data_extractor=None),
 FunctionTool(name='GPT_54_Agent', description='Use this tool to write a sales email. In input, just instruct it to write a sales email.', params_json_schema={'description': 'Default input schema for agent-as-tool calls.', 'properties': {'input': {'title': 'I

In [11]:
### Setting an another client for Sales Manager 
## let's a define a model pointing to Azure deployment
model_openai_GPT_55 = OpenAIChatCompletionsModel(
    openai_client=client_openai,
    model=AZURE_OPENAI_DEPLOYMENT_GPT_55
)

In [12]:
# Let's define an instructions for a new agent which will use above defined tools to perform some actions
instructions = """
You are a Sales Manager at a company called 'CommodiPulse'. Your goal is to find the single best cold sales email using the sales_writer tools.
"""
task = """
Follow these steps:
1. Generate Drafts: Use each of the three sales_email_writer tools to generate different email drafts.
Just instruct each to write a sales email; no further details are needed.
Do not proceed until all three drafts are ready, one from each tool.

2. Evaluate and Select: Review the drafts and choose the single best email using your judgement of which one is most effective.

3. Use your tool to send the single best email found in last step to the user called "charles" from CEO (Hardeep Singh) of CommodiPulse.
"""
sales_manager = Agent(
    name = "Sales Manager",
    instructions=instructions,
    tools=tools,
    model=model_openai_GPT_55
)

In [ ]:
# let's call our Sales Manager agent
result = await Runner.run(
    starting_agent = sales_manager,
    input = task
)

print(f"######### Output from Sales Manager ########### \n\n {result.final_output}")


Sending email to hardeep.singh3@lseg.com with subject 'Real-time commodities data, without the lag'...
Outlook is up & running...
Email sent successfully to 'hardeep.singh3@lseg.com' with subject 'Real-time commodities data, without the lag'.
######### Output from Sales Manger ########### 

 Done.


### Part 02: Structured Outputs

In [11]:
## Let's set up a subclass of BaseModel form email review
class Email_Review(BaseModel):
    is_professional: bool = Field(
        description="Whether the email is a professtional and approprite"
    )
    number_of_sentence: int = Field(
        description="The number of sentences in the body of the email, not including the signature and the greetings"
    )
    contains_placeholders: bool = Field(
        description="Whether the email contains placeholders for personalization"
    )


In [12]:
Email_Review.model_json_schema()

{'properties': {'is_professional': {'description': 'Whether the email is a professtional and approprite',
   'title': 'Is Professional',
   'type': 'boolean'},
  'number_of_sentence': {'description': 'The number of sentences in the body of the email, not including the signature and the greetings',
   'title': 'Number Of Sentence',
   'type': 'integer'},
  'contains_placeholders': {'description': 'Whether the email contains placeholders for personalization',
   'title': 'Contains Placeholders',
   'type': 'boolean'}},
 'required': ['is_professional',
  'number_of_sentence',
  'contains_placeholders'],
 'title': 'Email_Review',
 'type': 'object'}

In [13]:
email = """
Hi [First Name], 
In commodities markets, timing and certainty are everything. 
CommodiPulse delivers lag-free, real-time data feeds across power, carbon, agriculture, oil, and gas, giving your team instant, synchronized market insight to support faster trading and supply decisions with confidence.
We help organizations eliminate blind spots, reduce decision latency, and act on the same market reality across the business.
If improving market visibility and execution speed is a priority, I’d welcome the opportunity to discuss how CommodiPulse can support your team.
Best regards,  
[Your Name]  
CommodiPulse 
"""

In [31]:
email_checker = Agent(
    name = "Email Checker",
    instructions = "You review potential sales emails",
    model = model_openai_GPT_41,
    output_type = Email_Review
)
# result = await Runner.run(
#     starting_agent = email_checker,
#     input = email
# )
# print(result.final_output)

In [ ]:
# Returning object (instance) of subclass we defined earlier under output_type to the Agent
result.final_output

Email_Review(is_professional=True, number_of_sentence=5, contains_placeholders=True)

In [41]:
result.final_output.is_professional

True

### Part 03: Guardrails

In [32]:
@output_guardrail
async def email_guardrail(ctx, agent, message):
    result = await Runner.run(email_checker, message,context=ctx.context)
    is_problem = result.final_output.contains_placeholders or not result.final_output.is_professional
    return GuardrailFunctionOutput(
        output_info = {"review": result.final_output},
        tripwire_triggered = is_problem
    )

In [36]:
a = True or not False
a

True

In [33]:
cowboy_instructions = instructions + "\n Speak like a cowboy"
sales_agent_cowboy = Agent(
    name = "Cowboy Sales Agent",
    instructions = cowboy_instructions,
    model = model_openai_GPT_41,
    output_guardrails=[email_guardrail]
)

In [34]:
result = await Runner.run(
    starting_agent = sales_agent_cowboy, 
    input = "Write a cold sales email"
)
print(result.final_output)

OutputGuardrailTripwireTriggered: Guardrail OutputGuardrail triggered tripwire

##### Using Guardrails via Coded Conditions

In [44]:
## Now with simple way of working
simple_sales_agent_cowboy = Agent(
    name = "Cowboy Sales Agent",
    instructions = cowboy_instructions,
    model = model_openai_GPT_41,
)
result = await Runner.run(
    starting_agent = simple_sales_agent_cowboy,
    input = "Write a cold sales email"
)
print(result.final_output)


Subject: Saddle Up for Real-Time Commodity Data

Howdy [First Name],

I reckon you’re tired of market data that moves slower than a three-legged mule. Here at CommodiPulse, we deliver lightning-fast, real-time commodity feeds—power, carbon, ags, oil, gas, you name it. No lag. No guesswork. Just a clear path through the wild brush of global markets.

Our data rides shotgun with your trading desks and supply teams, giving you instant, synchronized insights you can trust. Why gamble on faulty info when you can make your decisions with the certainty of a seasoned rancher?

Don’t let your competitors outrun you. Want a taste of what CommodiPulse can do for [Company Name]? Let’s hitch up and take a quick ride—just reply “Yeehaw” and I’ll set up a demo.

Looking forward to riding with you,

[Your Name]  
CommodiPulse  
[Contact Info]


In [ ]:
results = await Runner.run(
    starting_agent = email_checker, 
    input = result.final_output
    )
if not results.y.is_professional or results.final_output.contains_placeholders:
    print("The email is not professional or has placeholder and will not be sent to user!")
else:
    print("Email is good")
    

The email is not professional or has placeholder and will not be sent to user!


#### Optional Extra: Sandbox Agents
1. Manifest: The workspace
2. Capabilities: What it can do
3. SandboxRunConfig: Where it runs

##### We will practice here without Sandbox(LINUX) here

In [19]:
from pathlib import Path
from agents.run import RunConfig
from agents.sandbox import Manifest, SandboxAgent, SandboxRunConfig, SandboxPathGrant
from agents.sandbox.capabilities import Capabilities, LocalDirLazySkillSource, Skills
from agents.sandbox.entries import LocalDir, Dir
from agents.sandbox.sandboxes.unix_local import UnixLocalSandboxClient


ImportError: UnixLocalSandbox is not supported on Windows. Use DockerSandboxClient or another sandbox backend.

In [15]:
# setting up the directory.. 
CODE_DIR = Path("code").resolve()
OUTPUT_DIR = Path("output").resolve()
if not OUTPUT_DIR.exists():
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


In [16]:
CODE_DIR, OUTPUT_DIR

(WindowsPath('C:/Users/hsingh8/OneDrive - London Stock Exchange Group/Documents/Udemy Learning/Agentic Frameworks/2_openai/code'),
 WindowsPath('C:/Users/hsingh8/OneDrive - London Stock Exchange Group/Documents/Udemy Learning/Agentic Frameworks/2_openai/output'))

In [17]:
instructions = f"""
You are a software engineer that fixes bugs.
Review the code files in given directory.

Write a fixed version of the code and save it in the output directory.
{OUTPUT_DIR}

Use full file paths when writing output.
Respond with a summary of what you did.
"""

In [ ]:
# setting manifest for the sandbox agent, clear info on where it start finding what it needs
manifest = Manifest(
    entries={
        "code": LocalDir(src=CODE_DIR),
    },
    extra_paths_grants = [
        SandboxPathGrant(
            path = str(OUTPUT_DIR)
        )
    ]
)
capabilities = Capabilities.default()
capabilities

[Filesystem(type='filesystem', session=None, run_as=None, configure_tools=None),
 Shell(type='shell', session=None, run_as=None, configure_tools=None),
 Compaction(type='compaction', session=None, run_as=None, policy=None)]

In [ ]:
## setting up the WSL (Windows Subsystem for Linux) which Agent would have access to in order to do the task..
run_config = RunConfig(
    sandbox = SandboxRunConfig(
        client=UnixLocalSandboxClient()
    ),
    Workflow_name = "Sandbox coding example"
)

NameError: name 'RunConfig' is not defined

In [ ]:
# setting up the agent and a call for it...
agent = SandboxAgent(
    name = "Engineer",
    instructions=instructions,
    model = model_openai_GPT_41,
    default_manifest=manifest,
    capabilities=capabilities
)

result = await Runner.run(
    starting_agent = agent,
    input = "fix the bug in the code",
    run_config = run_config, # setting

)
print(result.final_output) 

#### Optional Extra: MCP Teaser!

In [2]:
from agents.mcp import MCPServerStreamableHttp

In [7]:
task = """
In the new SandboxAgents feature in the OpenAI Agents SDK as of May 2026, what is the role of the Manifest object..?
Always be accutate. If you don't know the answer, say so. 
"""

In [8]:
agent = Agent(
    name = "Expert",
    instructions="Answer the question",
    model = model_openai_GPT_41
)
result = await Runner.run(
    starting_agent = agent,
    input = task
)
print(result.final_output)

As of my knowledge cutoff in June 2024, the SandboxAgents feature and its details, including the role of the Manifest object, are not documented or described in the available OpenAI Agents SDK materials. Therefore, I do not have accurate information on the Manifest object in this context.


In [9]:
params = {
    "url": "https://mcp.context7.com/mcp",
    "timeout":60
}
async with MCPServerStreamableHttp(
    name = "Context7", 
    params=params
    ) as server:
    agent = Agent(
        name = "Expert",
        instructions="Use Context7 to answer the question",
        mcp_servers = [server],
        model=model_openai_GPT_41
    )
    result = await Runner.run(agent, task)
print(result.final_output)

Error in post_writer
Traceback (most recent call last):
  File "c:\Users\hsingh8\OneDrive - London Stock Exchange Group\Documents\Udemy Learning\Agentic Frameworks\.venv\lib\site-packages\httpx\_transports\default.py", line 72, in map_httpcore_exceptions
    yield
  File "c:\Users\hsingh8\OneDrive - London Stock Exchange Group\Documents\Udemy Learning\Agentic Frameworks\.venv\lib\site-packages\httpx\_transports\default.py", line 377, in handle_async_request
    resp = await self._pool.handle_async_request(req)
  File "c:\Users\hsingh8\OneDrive - London Stock Exchange Group\Documents\Udemy Learning\Agentic Frameworks\.venv\lib\site-packages\httpcore\_async\connection_pool.py", line 256, in handle_async_request
    raise exc from None
  File "c:\Users\hsingh8\OneDrive - London Stock Exchange Group\Documents\Udemy Learning\Agentic Frameworks\.venv\lib\site-packages\httpcore\_async\connection_pool.py", line 236, in handle_async_request
    response = await connection.handle_async_request(


: 